# 🚀 6G Beam Prediction — Train on Google Colab (GPU)
**Project: ML-Based 6G Channel Modeling and Beam Prediction | VIT Chennai · BECE311L**

---
### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (`Ctrl+F9`)
3. Authorize Google Drive when prompted

---
### Scenario guide
| Scenario | Download | RAM during load | Expected Top-1 |
|----------|----------|-----------------|----------------|
| `asu_campus_3p5` ✅ | Auto (~150 MB zip) | ~135 MB | ~73% |
| `o1_28` | Auto (~300 MB zip, may hit rate limit) | ~185 MB | 88–92% |
| synthetic | None | <50 MB | ~1.6% (random) |

> **Recommendation:** Use `asu_campus_3p5` in Colab — it auto-downloads reliably and never hits rate limits. Use `o1_28` only if you have already uploaded the zip to Drive.

## ⚡ Step 0 — Verify GPU & Environment

In [ ]:
import sys, torch, psutil

print(f'Python : {sys.version.split()[0]}')
assert sys.version_info >= (3, 10), 'Python 3.10+ required'

if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM)')
    print(f'CUDA   : {torch.version.cuda} | PyTorch: {torch.__version__}')
else:
    raise SystemExit('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

ram_total = psutil.virtual_memory().total / 1e9
ram_avail = psutil.virtual_memory().available / 1e9
print(f'RAM    : {ram_avail:.1f} GB free / {ram_total:.1f} GB total')
print('✅ Environment ready')

## 📦 Step 1 — Install Dependencies

In [ ]:
print('Installing deepmimo...')
!pip install -q 'deepmimo==4.0.0b11' --pre
try:
    import deepmimo
    print(f'✅ deepmimo    : {deepmimo.__version__}')
except ImportError:
    print('Pinned version failed — trying latest pre-release...')
    !pip install -q 'deepmimo>=4.0.0b11' --pre
    import deepmimo
    print(f'✅ deepmimo    : {deepmimo.__version__} (latest)')

!pip install -q scikit-learn tqdm
import sklearn, tqdm
print(f'✅ scikit-learn: {sklearn.__version__}')
print(f'✅ tqdm        : {tqdm.__version__}')

## 💾 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/6G-ML'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'✅ Drive mounted → {DRIVE_OUTPUT_DIR}')

## ⚙️ Step 3 — Training Configuration

**Key memory-control settings:**
- `scenario`: Use `'asu_campus_3p5'` — auto-downloads, no rate limit, reliable ✅
- `max_users`: Hard cap on dataset size. `25_000` keeps RAM safely under 1 GB at all times.

In [ ]:
CONFIG = {
    # ── Dataset ──────────────────────────────────────────────────────────
    # 'asu_campus_3p5' → auto-downloads, 3.5 GHz, ~73% Top-1  ← RECOMMENDED for Colab
    # 'o1_28'          → may hit download rate limit; upload zip to Drive first if needed
    # False / use_real=False → synthetic data, no download, verifies training pipeline
    'scenario'       : 'asu_campus_3p5',
    'use_real'       : True,
    'n_synthetic'    : 18000,

    # ── Memory cap ───────────────────────────────────────────────────────
    # After loading all users, randomly subsample to at most max_users.
    # None = use the full dataset (may use more RAM).
    # 25_000 users × 64 antennas × 16 bytes = ~25 MB — completely safe.
    # Accuracy vs. full dataset: typically within 1–2% Top-1.
    'max_users'      : 25_000,

    # ── Model ────────────────────────────────────────────────────────────
    'input_dim'      : 6,
    'hidden_dims'    : [512, 256, 128],
    'num_classes'    : 64,
    'dropout'        : 0.2,
    'n_res_blocks'   : 2,

    # ── Training ─────────────────────────────────────────────────────────
    'epochs'         : 200,
    'batch_size'     : 512,
    'lr'             : 1e-3,
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.05,
    'log_every'      : 10,
    'seed'           : 42,

    # ── Hardware ─────────────────────────────────────────────────────────
    'use_amp'        : True,
    'grad_clip'      : 1.0,
    'device'         : 'cuda',

    # ── Paths ────────────────────────────────────────────────────────────
    'checkpoint_dir' : '/content/checkpoints',
    'checkpoint_name': 'best_mlp.pt',
    'history_name'   : 'training_history.json',
    'scenario_dir'   : '/content/deepmimo_scenarios',
}

print('✅ Configuration loaded:')
for k, v in CONFIG.items():
    print(f'   {k:<20} = {v}')

## 🧠 Step 4 — Embed Source Code

In [ ]:
import os
os.makedirs('/content/src', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

with open('/content/src/__init__.py', 'w') as f:
    f.write('')

MODEL_SRC = '''
import torch, torch.nn as nn, torch.nn.functional as F
from typing import List

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(dim); self.fc1 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim); self.fc2 = nn.Linear(dim, dim)
        self.drop = nn.Dropout(p=dropout)
        nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity=\'relu\')
        nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity=\'relu\')
        self.fc2.weight.data.mul_(0.01)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        r = x
        out = F.gelu(self.bn1(x))
        out = F.gelu(self.bn2(self.fc1(out)))
        return self.drop(self.fc2(out)) + r


class BeamPredictor(nn.Module):
    def __init__(self, input_dim=3, hidden_dims=None, num_classes=64,
                 dropout=0.2, n_res_blocks=2):
        super().__init__()
        if hidden_dims is None: hidden_dims = [512, 256, 128]
        stem_dim = hidden_dims[0]; neck_dims = hidden_dims[1:]
        self.stem  = nn.Sequential(nn.Linear(input_dim, stem_dim),
                                   nn.BatchNorm1d(stem_dim), nn.GELU())
        self.trunk = nn.Sequential(*[ResBlock(stem_dim, dropout/2) for _ in range(n_res_blocks)])
        layers, d = [], stem_dim
        for i, od in enumerate(neck_dims):
            layers += [nn.Linear(d, od), nn.BatchNorm1d(od), nn.GELU(),
                       nn.Dropout(dropout if i == 0 else dropout/2)]
            d = od
        self.neck = nn.Sequential(*layers)
        self.head  = nn.Linear(d, num_classes)
        self.config = {\'input_dim\': input_dim, \'hidden_dims\': hidden_dims,
                       \'num_classes\': num_classes, \'dropout\': dropout,
                       \'n_res_blocks\': n_res_blocks}
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if m.weight.requires_grad and m.weight.shape[0]:
                    try: nn.init.kaiming_uniform_(m.weight, nonlinearity=\'relu\')
                    except: pass
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.neck(self.trunk(self.stem(x))))


def count_parameters(model):
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n:,}"); return n
'''

with open('/content/src/model.py', 'w') as f:
    f.write(MODEL_SRC)
print('✅ model.py written')

In [ ]:
DATASET_SRC = '''
"""
dataset.py — Memory-safe DeepMIMO pipeline with max_users subsampling.

Memory fixes applied:
  #1  n_subcarriers=1 → 64x less RAM during channel computation
  #2  del channels_full immediately after slicing → no double-copy
  #3  channels=None to make_dataloaders → no ch_split copy
  #4  max_users random subsampling → caps dataset at any size
"""

import os, gc
from typing import Tuple, Optional
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def dft_codebook(N_t=64, N_b=64):
    si = np.arange(N_b)
    sv = 2.0 * si / N_b - 1.0
    ai = np.arange(N_t)
    return ((1.0/np.sqrt(N_t)) * np.exp(1j * np.pi * np.outer(sv, ai))).astype(np.complex128)


# FIX #1: n_subcarriers=1 for every scenario.
# DeepMIMO allocates (N, N_r, N_t, n_subcarriers) complex128 internally.
# 1 vs 64 subcarriers = 64x less RAM, zero accuracy impact (we use sc 0 only).
_SCENARIO_DEFAULTS = {
    \'o1_28\'         : {\'N_b_ant_shape\': [64,1], \'n_subcarriers\': 1, \'subcarrier_idx\': 0, \'num_paths\': 10, \'description\': \'Outdoor-1 28GHz\'},
    \'o1_28b\'        : {\'N_b_ant_shape\': [64,1], \'n_subcarriers\': 1, \'subcarrier_idx\': 0, \'num_paths\': 10, \'description\': \'Outdoor-1B 28GHz\'},
    \'asu_campus_3p5\': {\'N_b_ant_shape\': [64,1], \'n_subcarriers\': 1, \'subcarrier_idx\': 0, \'num_paths\': 10, \'description\': \'ASU Campus 3.5GHz\'},
}


def load_deepmimo_v4(scenario=\'asu_campus_3p5\', N_t=64, N_b_ant_shape=None,
                     n_subcarriers=1, subcarrier_idx=0, num_paths=10,
                     local_scenario_dir=None, max_users=None, seed=42):
    """
    Load a DeepMIMO scenario and return (positions, channels).
    max_users: if set, randomly subsample to this many users BEFORE
               returning, keeping peak memory bounded.
    """
    try:
        import deepmimo as dm
    except ImportError as e:
        raise ImportError("deepmimo not installed.") from e

    sc_key = scenario.lower()
    if sc_key in _SCENARIO_DEFAULTS:
        d = _SCENARIO_DEFAULTS[sc_key]
        if N_b_ant_shape is None: N_b_ant_shape = d[\'N_b_ant_shape\']
        n_subcarriers  = d[\'n_subcarriers\']
        subcarrier_idx = d[\'subcarrier_idx\']
        print(f"[DeepMIMO] {d[\'description\']}  n_subcarriers={n_subcarriers}  max_users={max_users}")
    else:
        if N_b_ant_shape is None: N_b_ant_shape = [N_t, 1]

    # Resolve local path
    project_root    = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    local_dir       = local_scenario_dir or os.path.join(project_root, \'deepmimo_scenarios\')
    local_path      = os.path.join(local_dir, sc_key)
    scenario_folder = dm.get_scenario_folder(scenario)

    if os.path.exists(local_path) and not os.path.exists(scenario_folder):
        import shutil
        os.makedirs(os.path.dirname(scenario_folder), exist_ok=True)
        target = os.path.join(os.path.dirname(scenario_folder), sc_key)
        if not os.path.exists(target):
            shutil.copytree(local_path, target)
            print(f"[DeepMIMO] Copied \'{scenario}\' from local dir to cache.")
        scenario_folder = target

    if not os.path.exists(scenario_folder):
        print(f"[DeepMIMO] Downloading \'{scenario}\' ...")
        result = dm.download(scenario)
        if result is None and not os.path.exists(dm.get_scenario_folder(scenario)):
            raise RuntimeError(
                f"Download failed for \'{scenario}\'.\n"
                "If using o1_28: upload the zip to Drive and extract to /content/deepmimo_scenarios/o1_28/\n"
                "Or switch to scenario=\'asu_campus_3p5\' which auto-downloads reliably.")

    print(f"[DeepMIMO v4] Loading \'{scenario}\' ...")
    dataset = dm.load(scenario)

    params = dm.ChannelParameters()
    params.bs_antenna[\'shape\']  = np.array(N_b_ant_shape)
    params.ue_antenna[\'shape\']  = np.array([1, 1])
    params.num_paths             = num_paths
    params.freq_domain           = 1
    params.ofdm[\'subcarriers\'] = n_subcarriers
    params.ofdm[\'selected_subcarriers\'] = np.array([subcarrier_idx])

    if len(dataset.datasets) > 1:
        print(f"[DeepMIMO v4] {len(dataset.datasets)} TX-RX pairs — keeping first only.")
        dataset.datasets = [dataset.datasets[0]]

    print(f"[DeepMIMO v4] Computing channels (BS={N_b_ant_shape}, sc={subcarrier_idx}/{n_subcarriers}) ...")
    dataset.compute_channels(params)

    # FIX #2: make a contiguous copy then immediately free the 4D original
    channels_full = np.array(dataset.channel)                    # (N, N_r, N_t, n_sc)
    positions     = np.array(dataset.rx_pos)                     # (N, 3)
    channels      = np.ascontiguousarray(channels_full[..., 0])  # (N, N_r, N_t)
    del channels_full, dataset
    gc.collect()

    print(f"[DeepMIMO v4] Raw: {channels.shape[0]:,} users | "
          f"channel {channels.shape[1:]} | {channels.nbytes/1e6:.1f} MB")

    # FIX #4: max_users random subsample
    if max_users is not None and len(channels) > max_users:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(channels), size=max_users, replace=False)
        idx.sort()
        channels  = np.ascontiguousarray(channels[idx])
        positions = positions[idx]
        gc.collect()
        print(f"[DeepMIMO v4] Subsampled to {max_users:,} users | "
              f"{channels.nbytes/1e6:.1f} MB")

    return positions, channels


def generate_synthetic_dataset(n_users=18000, N_t=64, N_r=1,
                               scene_size_m=500.0, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, scene_size_m, n_users)
    y = rng.uniform(0, scene_size_m, n_users)
    z = np.zeros(n_users)
    positions = np.stack([x, y, z], axis=1)
    real = rng.standard_normal((n_users, N_r, N_t)) / np.sqrt(2)
    imag = rng.standard_normal((n_users, N_r, N_t)) / np.sqrt(2)
    channels = (real + 1j*imag).astype(np.complex128)
    print(f"[Synthetic] {n_users:,} users | {channels.nbytes/1e6:.1f} MB")
    return positions, channels


def filter_valid_users(channels, positions, norm_threshold=1e-6):
    norms = np.linalg.norm(channels.reshape(channels.shape[0], -1), axis=1)
    mask  = norms > norm_threshold
    n_rm  = int((~mask).sum())
    if n_rm > 0:
        print(f"[Filter] Removed {n_rm:,} blocked users | remaining: {mask.sum():,}")
    return channels[mask], positions[mask]


def compute_beam_labels(channels, codebook):
    if channels.ndim == 3 and channels.shape[1] == 1:
        h = channels[:, 0, :]
    elif channels.ndim == 3:
        h = channels.reshape(channels.shape[0], -1)
        codebook = np.tile(codebook, (1, channels.shape[1]))
    else:
        h = channels
    bf_gain = np.abs(h @ codebook.conj().T) ** 2
    labels  = np.argmax(bf_gain, axis=1).astype(np.int64)
    print(f"[Labels] {len(np.unique(labels))} unique beams / {codebook.shape[0]} total")
    return labels


class BeamDataset(torch.utils.data.Dataset):
    def __init__(self, pos, labels):
        self.X = torch.tensor(pos,    dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def make_dataloaders(positions, labels, channels=None, batch_size=256,
                     val_ratio=0.10, test_ratio=0.20, seed=42, num_workers=2):
    N = len(labels)
    iv, it = train_test_split(np.arange(N), test_size=test_ratio,
                               stratify=labels, random_state=seed)
    itr, iv2 = train_test_split(iv, test_size=val_ratio/(1-test_ratio),
                                 stratify=labels[iv], random_state=seed)

    x, y, z  = positions[:,0], positions[:,1], positions[:,2]
    r         = np.sqrt(x**2 + y**2 + z**2)
    az        = np.arctan2(y, x)
    el        = np.arcsin(np.clip(z/(r+1e-6), -1, 1))
    feats     = np.stack([x,y,z,r,az,el], axis=1).astype(np.float64)
    feats     = np.nan_to_num(feats, nan=0, posinf=0, neginf=0)

    sc        = StandardScaler()
    fs        = feats.copy().astype(np.float32)
    fs[itr]   = sc.fit_transform(feats[itr])
    fs[iv2]   = sc.transform(feats[iv2])
    fs[it]    = sc.transform(feats[it])

    kw = dict(num_workers=num_workers, pin_memory=True)
    tl = DataLoader(BeamDataset(fs[itr], labels[itr]), batch_size, True,  **kw)
    vl = DataLoader(BeamDataset(fs[iv2], labels[iv2]), batch_size, False, **kw)
    el = DataLoader(BeamDataset(fs[it],  labels[it]),  batch_size, False, **kw)

    ch_split = {}
    if channels is not None:
        ch_split = {\'train\':channels[itr], \'val\':channels[iv2], \'test\':channels[it]}

    print(f"[DataLoader] train={len(tl.dataset):,} val={len(vl.dataset):,} test={len(el.dataset):,}")
    return tl, vl, el, sc, ch_split
'''

with open('/content/src/dataset.py', 'w') as f:
    f.write(DATASET_SRC)
print('✅ dataset.py written')

import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from src.model   import BeamPredictor, count_parameters
from src.dataset import (dft_codebook, generate_synthetic_dataset, load_deepmimo_v4,
                         filter_valid_users, compute_beam_labels, make_dataloaders)
print('✅ All modules imported')

## 📡 Step 5 — Load Dataset

RAM at each stage with default config (`asu_campus_3p5`, `max_users=25000`):

| Stage | Peak RAM |
|-------|----------|
| `compute_channels` (n_subcarriers=1) | ~135 MB |
| After `del channels_full` | ~135 MB |
| After `max_users=25000` subsample | **~25 MB** |
| After `del channels` (FIX #3) | **~5 MB** |

In [ ]:
import numpy as np, gc, os, psutil

def ram_gb(): return psutil.Process().memory_info().rss / 1e9

os.makedirs(CONFIG['scenario_dir'], exist_ok=True)
print(f'RAM before load : {ram_gb():.2f} GB')

if CONFIG['use_real']:
    print(f"Loading '{CONFIG['scenario']}' (max_users={CONFIG['max_users']}) ...")
    positions, channels = load_deepmimo_v4(
        scenario=CONFIG['scenario'],
        local_scenario_dir=CONFIG['scenario_dir'],
        max_users=CONFIG['max_users'],
        seed=CONFIG['seed'],
    )
else:
    print(f"Synthetic dataset ({CONFIG['n_synthetic']:,} users) ...")
    positions, channels = generate_synthetic_dataset(
        n_users=CONFIG['n_synthetic'], seed=CONFIG['seed'])

print(f'RAM after load  : {ram_gb():.2f} GB')

channels, positions = filter_valid_users(channels, positions)

n_users   = channels.shape[0]
ch_shape  = channels.shape
ch_mb     = channels.nbytes / 1e6
print(f'Dataset size    : {n_users:,} users | {ch_mb:.1f} MB')

# Beam labels
codebook = dft_codebook(N_t=CONFIG['num_classes'], N_b=CONFIG['num_classes'])
labels   = compute_beam_labels(channels, codebook)

# FIX #3: free channels — only positions + labels needed for training
del channels, codebook
gc.collect()
print(f'RAM after free  : {ram_gb():.2f} GB  ✅ channels freed')

train_loader, val_loader, test_loader, scaler, _ = make_dataloaders(
    positions, labels, channels=None,
    batch_size=CONFIG['batch_size'], seed=CONFIG['seed'], num_workers=2,
)

print()
print('✅ Data pipeline ready!')
print(f'   Users used   : {n_users:,} (of full scenario)')
print(f'   Channel shape: {ch_shape}  (freed from RAM)')
print(f'   Labels       : {len(np.unique(labels))} beam classes')
print(f'   Final RAM    : {ram_gb():.2f} GB')

## 🏗️ Step 6 — Build Model

In [ ]:
import torch, torch.nn as nn
import numpy as np

torch.manual_seed(CONFIG['seed']); np.random.seed(CONFIG['seed'])

device = torch.device(CONFIG['device'] if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model = BeamPredictor(
    input_dim=CONFIG['input_dim'], hidden_dims=CONFIG['hidden_dims'],
    num_classes=CONFIG['num_classes'], dropout=CONFIG['dropout'],
    n_res_blocks=CONFIG['n_res_blocks'],
).to(device)

count_parameters(model)

## 🔥 Step 7 — Train the Model

In [ ]:
import time, json, os
import torch.nn as nn

optimizer = torch.optim.Adam(model.parameters(),
                             lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-5)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

use_amp   = CONFIG['use_amp'] and device.type == 'cuda'
grad_clip = CONFIG['grad_clip']
print(f'AMP bfloat16: {"enabled" if use_amp else "disabled"}')

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
best_ckpt = os.path.join(CONFIG['checkpoint_dir'], CONFIG['checkpoint_name'])
history   = {'train_loss':[], 'val_loss':[], 'val_acc':[],
             'best_val_acc': 0.0, 'best_epoch': 0}


def run_epoch(model, loader, optimizer, criterion, device, grad_clip, use_amp, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(X); loss = criterion(logits, y)
            else:
                logits = model(X); loss = criterion(logits, y)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            total_loss += loss.item(); n += 1
            correct    += (logits.argmax(1) == y).sum().item(); total += len(y)
    return total_loss / max(n,1), correct / max(total,1)


print(f'\n{"═"*60}')
print(f'  {CONFIG["epochs"]} epochs | batch {CONFIG["batch_size"]} | {device} | {n_users:,} users')
print(f'{"═"*60}')

t0 = time.time()
for epoch in range(1, CONFIG['epochs']+1):
    tr_loss, _        = run_epoch(model, train_loader, optimizer, criterion, device, grad_clip, use_amp, True)
    val_loss, val_acc = run_epoch(model, val_loader,   None,      criterion, device, grad_clip, use_amp, False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > history['best_val_acc']:
        history['best_val_acc'] = val_acc
        history['best_epoch']   = epoch
        torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(), 'val_acc':val_acc,
                    'config':{**CONFIG, **model.config}}, best_ckpt)

    if epoch % CONFIG['log_every'] == 0 or epoch == 1:
        print(f"Ep {epoch:>3}/{CONFIG['epochs']} | "
              f"Loss {tr_loss:.4f}/{val_loss:.4f} | "
              f"Acc {val_acc*100:.2f}% | "
              f"Best {history['best_val_acc']*100:.2f}% | "
              f"LR {scheduler.get_last_lr()[0]:.2e} | "
              f"{time.time()-t0:.0f}s")

total_time = time.time() - t0
print(f'\n✅ Done in {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'   Best Val Acc: {history["best_val_acc"]*100:.2f}% @ epoch {history["best_epoch"]}')

history_path = os.path.join(CONFIG['checkpoint_dir'], CONFIG['history_name'])
with open(history_path, 'w') as f: json.dump(history, f, indent=2)
print(f'   Checkpoint  : {best_ckpt}')
print(f'   History     : {history_path}')

## 📊 Step 8 — Test Set Evaluation (Top-1 & Top-5)

In [ ]:
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded epoch {ckpt["epoch"]} (val {ckpt["val_acc"]*100:.2f}%)')

model.eval()
top1 = top5 = total = 0
with torch.no_grad():
    for X, y in test_loader:
        X, y  = X.to(device), y.to(device)
        logits = model(X)
        top1  += (logits.argmax(1) == y).sum().item()
        top5  += (logits.topk(5,1).indices == y.unsqueeze(1)).any(1).sum().item()
        total += len(y)

top1_acc, top5_acc = top1/total, top5/total

print(f'\n{"═"*38}')
print(f'  TEST SET ({total:,} samples)')
print(f'{"═"*38}')
print(f'  Top-1 : {top1_acc*100:.2f}%')
print(f'  Top-5 : {top5_acc*100:.2f}%')
print(f'{"═"*38}')

## 📈 Step 9 — Training Curves

In [ ]:
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

ep = list(range(1, len(history['train_loss'])+1))
D = '#0f0f1a'; P = '#1a1a2e'; C = '#00d4ff'; O = '#ff6b35'
G = '#39ff14'; GR = '#2a2a4a'; T = '#e0e0ff'

def sax(ax, title):
    ax.set_facecolor(P); ax.tick_params(colors=T, labelsize=9)
    ax.spines[:].set_color(GR); ax.xaxis.label.set_color(T); ax.yaxis.label.set_color(T)
    ax.set_title(title, color=T, fontsize=11, fontweight='bold', pad=8)
    ax.grid(True, color=GR, lw=0.5, alpha=0.7)

fig = plt.figure(figsize=(15,5)); fig.patch.set_facecolor(D)
gs  = gridspec.GridSpec(1,3, wspace=0.35)

ax1 = fig.add_subplot(gs[0]); sax(ax1, 'Loss')
ax1.plot(ep, history['train_loss'], C, lw=1.5, label='Train')
ax1.plot(ep, history['val_loss'],   O, lw=1.5, label='Val', ls='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy')
ax1.legend(facecolor=P, edgecolor=GR, labelcolor=T, fontsize=8)

ax2 = fig.add_subplot(gs[1]); sax(ax2, 'Val Accuracy')
ax2.plot(ep, [v*100 for v in history['val_acc']], G, lw=1.5)
ax2.axhline(history['best_val_acc']*100, color='gold', lw=1, ls=':',
            label=f'Best {history["best_val_acc"]*100:.1f}% @ep{history["best_epoch"]}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Top-1 Accuracy (%)')
ax2.legend(facecolor=P, edgecolor=GR, labelcolor=T, fontsize=8)

ax3 = fig.add_subplot(gs[2]); sax(ax3, 'Final Metrics')
vals  = [history['best_val_acc']*100, top1_acc*100, top5_acc*100]
bars  = ax3.bar(['Top-1\n(Val)','Top-1\n(Test)','Top-5\n(Test)'],
                vals, color=[C,G,O], edgecolor=GR, width=0.5)
for b, v in zip(bars, vals):
    ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
             f'{v:.1f}%', ha='center', color=T, fontsize=9, fontweight='bold')
ax3.set_ylim(0,108); ax3.set_ylabel('Accuracy (%)')

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
fig.suptitle(f'6G Beam Prediction | {CONFIG["scenario"]} | {n_users:,} users | '
             f'{CONFIG["epochs"]} epochs | {gpu}',
             color=T, fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout()

plot_path = os.path.join(CONFIG['checkpoint_dir'], 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor=D)
plt.show()
print(f'Plot saved → {plot_path}')

## ☁️ Step 10 — Save to Google Drive

In [ ]:
import shutil
from datetime import datetime

ts      = datetime.now().strftime('%Y%m%d_%H%M')
run_dir = os.path.join(DRIVE_OUTPUT_DIR, f'run_{CONFIG["scenario"]}_{ts}')
os.makedirs(run_dir, exist_ok=True)

for src, name in [(best_ckpt,    CONFIG['checkpoint_name']),
                  (history_path, CONFIG['history_name']),
                  (plot_path,    'training_curves.png')]:
    if os.path.exists(src):
        dst = os.path.join(run_dir, name)
        shutil.copy2(src, dst)
        print(f'  ✅ {name} ({os.path.getsize(dst)/1024:.0f} KB)')

print(f'\n✅ Saved to Drive: My Drive/6G-ML/run_{CONFIG["scenario"]}_{ts}/')
print(f'   Download best_mlp.pt → c:\\Projects\\6G-ML\\checkpoints\\')

## 📦 Step 11 (Optional) — Direct Browser Download

In [ ]:
from google.colab import files
files.download(best_ckpt)
files.download(history_path)
files.download(plot_path)
print('✅ Downloads triggered.')